# GPT-2 Activation Overhead Analysis (multiple profile runs)

This notebook loads runs under '../results' (or auto-detected) and compares baselines:

- modified_hf (`hf_modified`)
- modified_hf_hook (`hf_modified_hook`)
- modified_hf_hook_async (`hf_modified_hook_async`)
- hf_hook (`huggingface_hook`)

It visualizes Top‑K overhead vs `modified_hf`, and grouped categories (slice / events / rearrange / compute / d2d_copy).


In [ ]:
import os, json
from pathlib import Path
from collections import defaultdict, OrderedDict
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.dpi'] = 140
sns.set_style('whitegrid')

def detect_results_root():
    env = os.getenv('PROM_RESULTS_ROOT')
    cands = []
    if env: cands.append(Path(env))
    # Common relative candidates: repo root / HF_Prometheus dir / notebooks dir
    cands += [
        Path('HF_Prometheus') / 'results',
        Path('results'),
        Path.cwd() / 'HF_Prometheus' / 'results',
        Path.cwd() / 'results',
    ]
    for p in cands:
        try:
            if p.exists() and p.is_dir():
                return p
        except Exception:
            pass
    return Path('results')

# Prefer relative path from this notebook directory to HF_Prometheus/results
DEFAULT_ROOT = Path('..') / 'results'
RESULTS_ROOT = DEFAULT_ROOT if DEFAULT_ROOT.exists() else detect_results_root()
print('Using RESULTS_ROOT =', RESULTS_ROOT)

RUN_DIR_NAMES = [
    'HF_modified_decode_async',
    'HF_modified_decode_async_2',
    'HF_modified_decode_async_3',
    'HF_modified_decode_async_4',
    'HF_modified_decode_async_5',
    'HF_modified_decode_async_6',
]
# If defaults are missing, auto-detect run directories starting with HF_modified_decode_async
def _suffix_num(name):
    m = re.search(r'_(\d+)$', name)
    return int(m.group(1)) if m else 0
try:
    if not any((RESULTS_ROOT / Path(n)).exists() for n in RUN_DIR_NAMES):
        cands = [p.name for p in RESULTS_ROOT.iterdir() if p.is_dir() and p.name.startswith('HF_modified_decode_async')]
        if cands:
            RUN_DIR_NAMES = sorted(cands, key=_suffix_num)
            print('Auto-detected RUN_DIR_NAMES =', RUN_DIR_NAMES)
except Exception as e:
    print('Auto-detect run dirs failed:', e)

BASELINES = OrderedDict([
    ('hf_modified', 'modified_hf'),
    ('hf_modified_hook', 'modified_hf_hook'),
    ('hf_modified_hook_async', 'modified_hf_hook_async'),
    ('huggingface_hook', 'hf_hook'),
])

def find_trace_file(run_dir: Path, subdir: str):
    d = run_dir / subdir
    if not d.exists():
        return None
    files = sorted(d.glob('*.json'))
    if not files:
        return None
    return files[-1]

# Filter wrapper events when aggregating: user_annotation/gpu_user_annotation and PyTorch Profiler Trace
def aggregate_ops(trace):
    agg = defaultdict(float)
    total = 0.0
    for evt in trace.get('traceEvents', []):
        if evt.get('ph') != 'X':
            continue
        cat = evt.get('cat', '')
        name = evt.get('name', '')
        if cat in ('user_annotation', 'gpu_user_annotation'):
            continue
        if cat == 'Trace' and 'PyTorch Profiler' in name:
            continue
        dur = evt.get('dur')
        if dur is None:
            continue
        key = (cat, name)
        agg[key] += dur
        total += dur
    return total, agg

def load_runs():
    runs = []
    for name in RUN_DIR_NAMES:
        run_dir = RESULTS_ROOT / name
        if not run_dir.exists():
            continue
        tables = {}
        totals = {}
        for sub, label in BASELINES.items():
            p = find_trace_file(run_dir, sub)
            if p is None:
                continue
            try:
                trace = json.loads(p.read_text())
            except Exception as e:
                print('Failed to read', p, e)
                continue
            total, agg = aggregate_ops(trace)
            tables[label] = agg
            totals[label] = total / 1e6
        if tables:
            runs.append((name, tables, totals))
    return runs

runs = load_runs()
runs_names = [r[0] for r in runs]
runs_names


## Total duration per run (ms)


In [ ]:
rows = []
for run_name, tables, totals in runs:
    for label, ms in totals.items():
        rows.append({'run': run_name, 'baseline': label, 'ms': ms})
df = pd.DataFrame(rows)
if not df.empty:
    order = [b for b in ['modified_hf','modified_hf_hook','modified_hf_hook_async','hf_hook'] if b in df['baseline'].unique()]
    g = sns.catplot(data=df, x='baseline', y='ms', col='run', kind='bar', col_wrap=2, order=order, height=3)
    g.set_xticklabels(rotation=30)
    g.fig.suptitle('Total duration (ms)', y=1.02)
    plt.show()
else:
    print('No data')


## Top‑K overhead vs modified_hf
- Sync: modified_hf_hook − modified_hf (expected `aten::slice` heavy)
- Async: modified_hf_hook_async − modified_hf (events/copies/rearrange)


In [ ]:
def top_deltas(base: dict, to: dict, k=15):
    keys = set(base) | set(to)
    items=[]
    for key in keys:
        delta = (to.get(key,0.0)-base.get(key,0.0))/1e3
        if abs(delta)>=1.0:
            items.append((delta,key))
    items.sort(reverse=True)
    return items[:k]

def plot_top(run_name, diff, title):
    if not diff:
        print(f'[{run_name}] No data')
        return
    deltas=[x[0] for x in diff]
    labels=[f'{x[1][0]}:{x[1][1]}' for x in diff]
    fig, ax = plt.subplots(figsize=(8,4))
    ax.barh(labels[::-1], deltas[::-1])
    ax.set_xlabel('Δ ms (vs modified_hf)')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

for run_name, tables, totals in runs:
    base = tables.get('modified_hf')
    if not base:
        continue
    sync = tables.get('modified_hf_hook')
    if sync:
        diff = top_deltas(base, sync, 20)
        plot_top(run_name, diff, f'{run_name} Sync Top‑K')
    async_tab = tables.get('modified_hf_hook_async')
    if async_tab:
        diff = top_deltas(base, async_tab, 20)
        plot_top(run_name, diff, f'{run_name} Async Top‑K')


## Grouped categories: slice / events / rearrange / compute / d2d_copy


In [ ]:
GROUPS = OrderedDict({
    'slice': ['aten::slice'],
    'events': ['cudaEventRecordWithFlags','cudaStreamWaitEvent','cudaStreamIsCapturing'],
    'tensor_rearrange': ['aten::reshape','aten::view','aten::empty','aten::clone','aten::copy_','aten::as_strided'],
    'compute_heads': ['aten::addmm','aten::matmul','aten::native_layer_norm','aten::layer_norm','aten::bmm'],
    'd2d_copy': ['cudaMemcpyAsync','Memcpy DtoD (Device -> Device)'],
})

def group_delta(base, to):
    out={g:0.0 for g in GROUPS}
    for (cat,name), dur in to.items():
        delta=(dur-base.get((cat,name),0.0))/1e3
        for g,keys in GROUPS.items():
            if any(name.startswith(k) for k in keys):
                out[g]+=delta
                break
    return out

rows=[]
for run_name, tables, totals in runs:
    base=tables.get('modified_hf')
    if not base: continue
    for tgt in ['modified_hf_hook','modified_hf_hook_async','hf_hook']:
        tab=tables.get(tgt)
        if not tab: continue
        gd=group_delta(base, tab)
        for g,val in gd.items():
            rows.append({'run':run_name,'target':tgt,'group':g,'delta_ms':val})
grp_df=pd.DataFrame(rows)
if not grp_df.empty:
    g=sns.catplot(data=grp_df, x='group', y='delta_ms', hue='target', col='run', kind='bar', col_wrap=2, height=3, sharex=False, sharey=False)
    g.set_xticklabels(rotation=45)
    g.fig.suptitle('Grouped overhead vs modified_hf (ms)', y=1.02)
    plt.show()
else:
    print('No grouped data')
